In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive
!mkdir -p vk_lsvd_project/{src,data/{interim,processed,mappings},outputs/{metrics,plots,logs},report}
!ls -R vk_lsvd_project

/content/drive/MyDrive
vk_lsvd_project:
data  outputs  report  src

vk_lsvd_project/data:
interim  mappings  metadata  processed

vk_lsvd_project/data/interim:
interactions_sample.parquet

vk_lsvd_project/data/mappings:
item2idx.json  user2idx.json

vk_lsvd_project/data/metadata:
metadata

vk_lsvd_project/data/metadata/metadata:
item_embeddings.npz  items_metadata.parquet  users_metadata.parquet

vk_lsvd_project/data/processed:
local_item_embeddings.npy  test.parquet  train.parquet	val.parquet

vk_lsvd_project/outputs:
logs  metrics  plots

vk_lsvd_project/outputs/logs:

vk_lsvd_project/outputs/metrics:
results_main.csv

vk_lsvd_project/outputs/plots:
ndcg10_bar.png	recall50_bar.png

vk_lsvd_project/report:

vk_lsvd_project/src:
build_splits.py  prepare_data.py	  train_eval_bpr.py
make_plots.py	 train_eval_baselines.py  train_eval_rerank.py


In [ ]:
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba --version

bin/micromamba
2.5.0


In [ ]:
!./bin/micromamba create -y -n recsys -c conda-forge python=3.10 \
  numpy pandas pyarrow scipy tqdm pyyaml matplotlib \
  implicit lightfm datasets

[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.3s
conda-forge/linux-64   2%
conda-forge/noarch    16%[+] 0.4s
conda-forge/linux-64   8%
conda-forge/noarch    28%[+] 0.5s
conda-forge/linux-64  12%
conda-forge/noarch    37%[+] 0.6s
conda-forge/linux-64  14%
conda-forge/noarch    43%[+] 0.7s
conda-forge/linux-64  17%
conda-forge/noarch    50%[+] 0.8s
conda-forge/linux-64  22%
conda-forge/noarch    55%[+] 0.9s
conda-forge/linux-64  26%
conda-forge/noarch    65%[+] 1.0s
conda-forge/linux-64  31%
conda-forge/noarch    75%[+] 1.1s
conda-forge/linux-64  37%
conda-forge/noarch    87%[+] 1.2s
conda-forge/linux-64  39%
conda-forge/noarch    94%[+] 1.3s
conda-forge/linux-64  42%
conda-forge/noarch   100%conda-forge/noarch                                
[+] 1.4s
conda-forge/linux-64  49%[+] 1.5s
conda-forge/linux-64  56%[+] 1.6s
conda-forge/linux-64  63%[+] 1.7s
conda-forge/linux-64  70%[+] 1.8s
conda-forge/lin

In [ ]:
!./bin/micromamba run -n recsys python -c "import sys; print(sys.version)"
!./bin/micromamba run -n recsys python -c "import implicit, lightfm, datasets; print('OK')"

3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]
OK


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/prepare_data.py << 'PY'
import os
import argparse
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--out", default="/content/drive/MyDrive/vk_lsvd_project/data/interim/interactions_sample.parquet")
    p.add_argument("--n", type=int, default=10000)
    args = p.parse_args()

    os.makedirs(os.path.dirname(args.out), exist_ok=True)

    ds = load_dataset("deepvk/VK-LSVD", split="train", streaming=True)

    rows = []
    shown = False

    for idx, x in enumerate(tqdm(ds)):
        if not shown:
            print("Available keys:", list(x.keys()))
            shown = True

        user = x.get("user_id")
        item = x.get("item_id")
        timespent = x.get("timespent", 0)

        if user is None or item is None:
            continue

        positive = int(timespent) >= 20
        if not positive:
            continue

        ts = idx
        rows.append((int(user), int(item), int(ts)))

        if len(rows) % 2000 == 0:
            print("positives collected:", len(rows))

        if len(rows) >= args.n:
            break

    df = pd.DataFrame(rows, columns=["user_id", "item_id", "ts"])
    df.to_parquet(args.out, index=False)
    print("Saved:", args.out, "rows:", len(df))
    print(df.head())

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/prepare_data.py --n 50000

README.md: 9.94kB [00:00, 11.3MB/s]
Resolving data files: 100% 225/225 [00:00<00:00, 12777.64it/s]
0it [00:00, ?it/s]Available keys: ['user_id', 'item_id', 'place', 'platform', 'agent', 'timespent', 'like', 'dislike', 'share', 'bookmark', 'click_on_author', 'open_comments']
5447it [00:06, 992.32it/s] positives collected: 2000
10682it [00:10, 1155.55it/s]positives collected: 4000
15936it [00:14, 1420.85it/s]positives collected: 6000
21143it [00:17, 1739.23it/s]positives collected: 8000
26224it [00:19, 2314.39it/s]positives collected: 10000
30931it [00:20, 3379.35it/s]positives collected: 12000
36201it [00:24, 1603.39it/s]positives collected: 14000
41187it [00:28, 1479.27it/s]positives collected: 16000
46875it [00:32, 1390.94it/s]positives collected: 18000
52643it [00:35, 1538.76it/s]positives collected: 20000
58126it [00:37, 3214.34it/s]positives collected: 22000
64286it [00:42, 1559.32it/s]positives collected: 24000
69941it [00:44, 3365.36it/s]positives collected: 26000
76125it [00:46,

In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/data/interim/

total 931K
-rw------- 1 root root 931K Feb 25 17:42 interactions_sample.parquet


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/build_splits.py << 'PY'
import os, json
import pandas as pd

INPUT = "/content/drive/MyDrive/vk_lsvd_project/data/interim/interactions_sample.parquet"
OUTDIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
MAPDIR = "/content/drive/MyDrive/vk_lsvd_project/data/mappings"

def main():
    os.makedirs(OUTDIR, exist_ok=True)
    os.makedirs(MAPDIR, exist_ok=True)

    df = pd.read_parquet(INPUT).sort_values(["user_id", "ts"]).reset_index(drop=True)

    train_parts = []
    val_parts = []
    test_parts = []

    for user_id, g in df.groupby("user_id", sort=False):
        g = g.sort_values("ts")
        n = len(g)

        if n == 1:
            train_parts.append(g)
        elif n == 2:
            train_parts.append(g.iloc[:1])
            test_parts.append(g.iloc[1:])
        else:
            train_parts.append(g.iloc[:-2])
            val_parts.append(g.iloc[-2:-1])
            test_parts.append(g.iloc[-1:])

    train = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame(columns=df.columns)
    val = pd.concat(val_parts, ignore_index=True) if val_parts else pd.DataFrame(columns=df.columns)
    test = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame(columns=df.columns)

    # users/items только из train
    users = pd.Index(train["user_id"].unique())
    items = pd.Index(train["item_id"].unique())

    user2idx = {int(u): i for i, u in enumerate(users)}
    item2idx = {int(it): i for i, it in enumerate(items)}

    def map_df(d):
        if len(d) == 0:
            return pd.DataFrame(columns=["u", "i", "ts"])
        d = d[d["user_id"].isin(user2idx) & d["item_id"].isin(item2idx)].copy()
        if len(d) == 0:
            return pd.DataFrame(columns=["u", "i", "ts"])
        d["u"] = d["user_id"].map(user2idx)
        d["i"] = d["item_id"].map(item2idx)
        return d[["u", "i", "ts"]]

    train_m = map_df(train)
    val_m = map_df(val)
    test_m = map_df(test)

    train_m.to_parquet(os.path.join(OUTDIR, "train.parquet"), index=False)
    val_m.to_parquet(os.path.join(OUTDIR, "val.parquet"), index=False)
    test_m.to_parquet(os.path.join(OUTDIR, "test.parquet"), index=False)

    with open(os.path.join(MAPDIR, "user2idx.json"), "w") as f:
        json.dump({str(k): v for k, v in user2idx.items()}, f)
    with open(os.path.join(MAPDIR, "item2idx.json"), "w") as f:
        json.dump({str(k): v for k, v in item2idx.items()}, f)

    user_hist = df.groupby("user_id").size()
    print("rows total:", len(df))
    print("users total:", df['user_id'].nunique())
    print("users with >=2 events:", int((user_hist >= 2).sum()))
    print("users with >=3 events:", int((user_hist >= 3).sum()))
    print("train/val/test after mapping:", len(train_m), len(val_m), len(test_m))
    print("mapped users/items:", len(user2idx), len(item2idx))

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/build_splits.py

rows total: 50000
users total: 43780
users with >=2 events: 5901
users with >=3 events: 245
train/val/test after mapping: 43854 92 2416
mapped users/items: 43780 29911


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/data/processed/

total 8.1M
-rw------- 1 root root 7.4M Feb 25 09:46 local_item_embeddings.npy
-rw------- 1 root root  41K Feb 25 17:42 test.parquet
-rw------- 1 root root 734K Feb 25 17:42 train.parquet
-rw------- 1 root root 3.7K Feb 25 17:42 val.parquet


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/train_eval_baselines.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def build_gt(df):
    gt = {}
    if len(df) == 0:
        return gt
    for u, g in df.groupby("u"):
        gt[int(u)] = set(map(int, g["i"].values))
    return gt

def main():
    os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)

    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

    if len(train) == 0:
        print("Train is empty. Stop.")
        return

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    gt = build_gt(test)

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = set(map(int, g["i"].values))

    users_eval = sorted(gt.keys())
    if len(users_eval) == 0:
        print("No users in test after mapping. Stop.")
        return

    # ---------- Popularity ----------
    pop = np.asarray(X.sum(axis=0)).ravel()
    pop_rank = np.argsort(-pop)

    def recommend_pop(u, k=100):
        seen = hist.get(u, set())
        rec = [i for i in pop_rank if i not in seen]
        return rec[:k]

    # ---------- ALS ----------
    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01)
    model.fit(X)  # <-- ВАЖНО: без .T

    def recommend_als(u, k=100):
        ids, _ = model.recommend(
            userid=u,
            user_items=X[u],
            N=k,
            filter_already_liked_items=True
        )
        return list(map(int, ids))

    rows = []

    for name, rec_fn in [("popularity", recommend_pop), ("als", recommend_als)]:
        r10, n10, r50, n50 = [], [], [], []
        for u in users_eval:
            rec = rec_fn(u, 100)
            gts = gt[u]
            r10.append(recall_at_k(rec, gts, 10))
            n10.append(ndcg_at_k(rec, gts, 10))
            r50.append(recall_at_k(rec, gts, 50))
            n50.append(ndcg_at_k(rec, gts, 50))

        rows.append({
            "model": name,
            "content": "none",
            "integration": "none",
            "recall@10": float(np.nanmean(r10)),
            "ndcg@10": float(np.nanmean(n10)),
            "recall@50": float(np.nanmean(r50)),
            "ndcg@50": float(np.nanmean(n50)),
            "users_eval": len(users_eval)
        })

    out_df = pd.DataFrame(rows)
    out_df.to_csv(OUT_FILE, index=False)
    print(out_df)

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/train_eval_baselines.py

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:06<00:00,  2.45it/s]
        model  ... users_eval
0  popularity  ...       2416
1         als  ...       2416

[2 rows x 8 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv

model,content,integration,recall@10,ndcg@10,recall@50,ndcg@50,users_eval
popularity,none,none,0.02566225165562914,0.013880355355822693,0.0658112582781457,0.02246150176082847,2416
als,none,none,0.004966887417218543,0.002869619585727842,0.021109271523178808,0.006350435404347251,2416


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/train_eval_bpr.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def build_gt(df):
    gt = {}
    if len(df) == 0:
        return gt
    for u, g in df.groupby("u"):
        gt[int(u)] = set(map(int, g["i"].values))
    return gt

def main():
    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

    if len(train) == 0:
        print("Train is empty. Stop.")
        return

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    gt = build_gt(test)
    users_eval = sorted(gt.keys())

    if len(users_eval) == 0:
        print("No users in test after mapping. Stop.")
        return

    model = BayesianPersonalizedRanking(
        factors=64,
        iterations=30,
        learning_rate=0.05,
        regularization=0.01
    )
    model.fit(X)  # <-- ВАЖНО: без .T

    r10, n10, r50, n50 = [], [], [], []

    for u in users_eval:
        ids, _ = model.recommend(
            userid=u,
            user_items=X[u],
            N=100,
            filter_already_liked_items=True
        )
        rec = list(map(int, ids))
        gts = gt[u]

        r10.append(recall_at_k(rec, gts, 10))
        n10.append(ndcg_at_k(rec, gts, 10))
        r50.append(recall_at_k(rec, gts, 50))
        n50.append(ndcg_at_k(rec, gts, 50))

    row = {
        "model": "bpr",
        "content": "none",
        "integration": "none",
        "recall@10": float(np.nanmean(r10)),
        "ndcg@10": float(np.nanmean(n10)),
        "recall@50": float(np.nanmean(r50)),
        "ndcg@50": float(np.nanmean(n50)),
        "users_eval": len(users_eval)
    }

    os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)

    if os.path.exists(OUT_FILE):
        df = pd.read_csv(OUT_FILE)
        df = df[df["model"] != "bpr"]  # чтобы не дублировать старую строку
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    df.to_csv(OUT_FILE, index=False)
    print(pd.DataFrame([row]))

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/train_eval_bpr.py

100% 30/30 [00:00<00:00, 78.14it/s, train_auc=54.02%, skipped=0.01%]
  model  ... users_eval
0   bpr  ...       2416

[1 rows x 8 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv

model,content,integration,recall@10,ndcg@10,recall@50,ndcg@50,users_eval
popularity,none,none,0.0256622516556291,0.0138803553558226,0.0658112582781457,0.0224615017608284,2416
als,none,none,0.0049668874172185,0.0028696195857278,0.0211092715231788,0.0063504354043472,2416
bpr,none,none,0.0004139072847682119,0.00014743674963080389,0.0016556291390728477,0.0004488578306860568,2416


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/make_plots.py << 'PY'
import os
os.environ["MPLBACKEND"] = "Agg"

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

INPUT = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv"
OUTDIR = "/content/drive/MyDrive/vk_lsvd_project/outputs/plots"

def plot_metric(df, metric, outpath):
    d = df.copy()
    d["label"] = d["model"] + "|" + d["content"] + "|" + d["integration"]
    d = d.sort_values(metric, ascending=False)

    plt.figure()
    plt.bar(d["label"], d[metric])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def main():
    os.makedirs(OUTDIR, exist_ok=True)

    df = pd.read_csv(INPUT)

    plot_metric(df, "ndcg@10", os.path.join(OUTDIR, "ndcg10_bar.png"))
    plot_metric(df, "recall@50", os.path.join(OUTDIR, "recall50_bar.png"))

    print("Plots saved to:", OUTDIR)

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/make_plots.py

Plots saved to: /content/drive/MyDrive/vk_lsvd_project/outputs/plots


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/outputs/plots/

total 70K
-rw------- 1 root root 36K Feb 25 17:42 ndcg10_bar.png
-rw------- 1 root root 34K Feb 25 17:42 recall50_bar.png


In [ ]:
from datasets import get_dataset_config_names
print(get_dataset_config_names("deepvk/VK-LSVD"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

['default']


In [ ]:
from datasets import load_dataset
ds = load_dataset("deepvk/VK-LSVD", split="train", streaming=True)
first = next(iter(ds))
print(first.keys())

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

dict_keys(['user_id', 'item_id', 'place', 'platform', 'agent', 'timespent', 'like', 'dislike', 'share', 'bookmark', 'click_on_author', 'open_comments'])


In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("deepvk/VK-LSVD", repo_type="dataset")
print("Total files:", len(files))
for f in files[:200]:
    print(f)

Total files: 244
.gitattributes
README.md
interactions/test/week_26.parquet
interactions/train/week_00.parquet
interactions/train/week_01.parquet
interactions/train/week_02.parquet
interactions/train/week_03.parquet
interactions/train/week_04.parquet
interactions/train/week_05.parquet
interactions/train/week_06.parquet
interactions/train/week_07.parquet
interactions/train/week_08.parquet
interactions/train/week_09.parquet
interactions/train/week_10.parquet
interactions/train/week_11.parquet
interactions/train/week_12.parquet
interactions/train/week_13.parquet
interactions/train/week_14.parquet
interactions/train/week_15.parquet
interactions/train/week_16.parquet
interactions/train/week_17.parquet
interactions/train/week_18.parquet
interactions/train/week_19.parquet
interactions/train/week_20.parquet
interactions/train/week_21.parquet
interactions/train/week_22.parquet
interactions/train/week_23.parquet
interactions/train/week_24.parquet
interactions/validation/week_25.parquet
metadata/

In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "deepvk/VK-LSVD"
local_dir = "/content/drive/MyDrive/vk_lsvd_project/data/metadata"

import os
os.makedirs(local_dir, exist_ok=True)

emb_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename="metadata/item_embeddings.npz",
    local_dir=local_dir
)

items_meta_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename="metadata/items_metadata.parquet",
    local_dir=local_dir
)

users_meta_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename="metadata/users_metadata.parquet",
    local_dir=local_dir
)

print(emb_path)
print(items_meta_path)
print(users_meta_path)

/content/drive/MyDrive/vk_lsvd_project/data/metadata/metadata/item_embeddings.npz
/content/drive/MyDrive/vk_lsvd_project/data/metadata/metadata/items_metadata.parquet
/content/drive/MyDrive/vk_lsvd_project/data/metadata/metadata/users_metadata.parquet


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/data/metadata

total 4.0K
drwx------ 2 root root 4.0K Feb 25 09:34 metadata


In [ ]:
import numpy as np

base = "/content/drive/MyDrive/vk_lsvd_project/data/metadata/metadata"

data = np.load(f"{base}/item_embeddings.npz", allow_pickle=True)

meta_item_ids = data["item_id"]
meta_embeddings = data["embedding"]

print("meta_item_ids shape:", meta_item_ids.shape, meta_item_ids.dtype)
print("meta_embeddings shape:", meta_embeddings.shape, meta_embeddings.dtype)
print("first item_id:", meta_item_ids[0])
print("first embedding shape:", np.array(meta_embeddings[0]).shape)

meta_item_ids shape: (19627601,) uint32
meta_embeddings shape: (19627601, 64) float16
first item_id: 38
first embedding shape: (64,)


In [ ]:
import json
import numpy as np

# локальный mapping из твоего train
with open("/content/drive/MyDrive/vk_lsvd_project/data/mappings/item2idx.json", "r") as f:
    item2idx_local = json.load(f)

# item_id -> row в metadata
meta_map = {int(item_id): idx for idx, item_id in enumerate(meta_item_ids)}

n_local_items = max(map(int, item2idx_local.values())) + 1
emb_dim = np.array(meta_embeddings[0]).shape[0]

local_emb = np.zeros((n_local_items, emb_dim), dtype=np.float32)

matched = 0
for raw_item_id_str, local_i in item2idx_local.items():
    raw_item_id = int(raw_item_id_str)
    if raw_item_id in meta_map:
        row = meta_map[raw_item_id]
        local_emb[int(local_i)] = np.array(meta_embeddings[row], dtype=np.float32)
        matched += 1

print("matched items:", matched, "of", len(item2idx_local))
print("local_emb shape:", local_emb.shape)

np.save("/content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_embeddings.npy", local_emb)
print("saved local_item_embeddings.npy")

matched items: 29911 of 29911
local_emb shape: (29911, 64)
saved local_item_embeddings.npy


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_embeddings.npy

-rw------- 1 root root 7.4M Feb 25 17:44 /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_embeddings.npy


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/train_eval_rerank.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def cosine(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def build_gt(df):
    gt = {}
    if len(df) == 0:
        return gt
    for u, g in df.groupby("u"):
        gt[int(u)] = set(map(int, g["i"].values))
    return gt

def main():
    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
    item_emb = np.load(os.path.join(DATA_DIR, "local_item_embeddings.npy"))

    if len(train) == 0:
        print("Train is empty. Stop.")
        return

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    gt = build_gt(test)
    users_eval = sorted(gt.keys())
    if len(users_eval) == 0:
        print("No users in test after mapping. Stop.")
        return

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = list(map(int, g["i"].values))

    user_profiles = {}
    for u, items in hist.items():
        vecs = item_emb[items]
        user_profiles[u] = vecs.mean(axis=0) if len(vecs) else np.zeros(item_emb.shape[1], dtype=np.float32)

    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01)
    model.fit(X)

    rows = []

    for alpha in [0.8, 0.5]:
        r10, n10, r50, n50 = [], [], [], []

        for u in users_eval:
            ids, scores = model.recommend(
                userid=u,
                user_items=X[u],
                N=200,
                filter_already_liked_items=True
            )

            profile = user_profiles[u]
            rescored = []

            for item_id, als_score in zip(ids, scores):
                sim = cosine(profile, item_emb[int(item_id)])
                final_score = alpha * float(als_score) + (1 - alpha) * sim
                rescored.append((int(item_id), final_score))

            rescored.sort(key=lambda x: x[1], reverse=True)
            rec = [item for item, _ in rescored]

            gts = gt[u]
            r10.append(recall_at_k(rec, gts, 10))
            n10.append(ndcg_at_k(rec, gts, 10))
            r50.append(recall_at_k(rec, gts, 50))
            n50.append(ndcg_at_k(rec, gts, 50))

        rows.append({
            "model": "als_rerank",
            "content": "item_emb",
            "integration": f"rerank_alpha_{alpha}",
            "recall@10": float(np.nanmean(r10)),
            "ndcg@10": float(np.nanmean(n10)),
            "recall@50": float(np.nanmean(r50)),
            "ndcg@50": float(np.nanmean(n50)),
            "users_eval": len(users_eval)
        })

    if os.path.exists(OUT_FILE):
        df = pd.read_csv(OUT_FILE)
        df = df[df["model"] != "als_rerank"]
        df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
    else:
        df = pd.DataFrame(rows)

    df.to_csv(OUT_FILE, index=False)
    print(pd.DataFrame(rows))

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/train_eval_rerank.py
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:09<00:00,  1.54it/s]
        model  ... users_eval
0  als_rerank  ...       2416
1  als_rerank  ...       2416

[2 rows x 8 columns]
model,content,integration,recall@10,ndcg@10,recall@50,ndcg@50,users_eval
popularity,none,none,0.0256622516556291,0.0138803553558226,0.0658112582781457,0.0224615017608284,2416
als,none,none,0.0049668874172185,0.0028696195857278,0.0211092715231788,0.0063504354043472,2416
bpr,none,none,0.0004139072847682,0.0001474367496308,0.0016556291390728,0.000448857830686,2416
als_rerank,item_emb,rerank_alpha_0.8,0.006208609271523178,0.0

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/make_plots.py

Plots saved to: /content/drive/MyDrive/vk_lsvd_project/outputs/plots


In [ ]:
import numpy as np
import pandas as pd
import json

base = "/content/drive/MyDrive/vk_lsvd_project/data/metadata/metadata"
items_meta = pd.read_parquet(f"{base}/items_metadata.parquet")

with open("/content/drive/MyDrive/vk_lsvd_project/data/mappings/item2idx.json", "r") as f:
    item2idx_local = json.load(f)

local_item_ids = set(map(int, item2idx_local.keys()))

meta = items_meta[items_meta["item_id"].isin(local_item_ids)].copy()
print("matched metadata rows:", len(meta), "of", len(local_item_ids))

# Числовые признаки
meta["duration"] = meta["duration"].fillna(0).astype(float)
meta["train_interactions_rank"] = meta["train_interactions_rank"].fillna(0).astype(float)

meta["duration_norm"] = (meta["duration"] - meta["duration"].mean()) / (meta["duration"].std() + 1e-8)
meta["rank_log"] = np.log1p(meta["train_interactions_rank"])
meta["rank_norm"] = (meta["rank_log"] - meta["rank_log"].mean()) / (meta["rank_log"].std() + 1e-8)

# author bucket: 128 корзин
meta["author_bucket"] = (meta["author_id"].fillna(0).astype(np.int64) % 128).astype(int)

n_local_items = max(map(int, item2idx_local.values())) + 1
meta_dim = 2 + 128  # duration_norm + rank_norm + author one-hot

local_meta = np.zeros((n_local_items, meta_dim), dtype=np.float32)

for _, row in meta.iterrows():
    raw_item_id = int(row["item_id"])
    local_i = int(item2idx_local[str(raw_item_id)])

    vec = np.zeros(meta_dim, dtype=np.float32)
    vec[0] = float(row["duration_norm"])
    vec[1] = float(row["rank_norm"])
    vec[2 + int(row["author_bucket"])] = 1.0

    local_meta[local_i] = vec

out_path = "/content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_metadata_features.npy"
np.save(out_path, local_meta)

print("saved:", out_path)
print("shape:", local_meta.shape)

matched metadata rows: 29911 of 29911
saved: /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_metadata_features.npy
shape: (29911, 130)


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_metadata_features.npy

-rw------- 1 root root 15M Feb 25 17:45 /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_metadata_features.npy


In [ ]:
import numpy as np

item_emb = np.load("/content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_embeddings.npy")
meta_feat = np.load("/content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_metadata_features.npy")

print("item_emb shape:", item_emb.shape)
print("meta_feat shape:", meta_feat.shape)

fused = np.concatenate([item_emb, meta_feat], axis=1).astype(np.float32)

# L2-нормализация по строкам
norms = np.linalg.norm(fused, axis=1, keepdims=True)
norms[norms == 0] = 1.0
fused = fused / norms

out_path = "/content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_fused_embeddings.npy"
np.save(out_path, fused)

print("saved:", out_path)
print("shape:", fused.shape)

item_emb shape: (29911, 64)
meta_feat shape: (29911, 130)
saved: /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_fused_embeddings.npy
shape: (29911, 194)


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_fused_embeddings.npy

-rw------- 1 root root 23M Feb 25 17:46 /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_fused_embeddings.npy


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/train_eval_rerank_generic.py << 'PY'
import os
import argparse
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def cosine(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def build_gt(df):
    gt = {}
    if len(df) == 0:
        return gt
    for u, g in df.groupby("u"):
        gt[int(u)] = set(map(int, g["i"].values))
    return gt

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--emb_path", required=True)
    p.add_argument("--content_name", required=True)
    args = p.parse_args()

    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
    item_emb = np.load(args.emb_path)

    if len(train) == 0:
        print("Train is empty. Stop.")
        return

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    gt = build_gt(test)
    users_eval = sorted(gt.keys())
    if len(users_eval) == 0:
        print("No users in test after mapping. Stop.")
        return

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = list(map(int, g["i"].values))

    user_profiles = {}
    for u, items in hist.items():
        vecs = item_emb[items]
        user_profiles[u] = vecs.mean(axis=0) if len(vecs) else np.zeros(item_emb.shape[1], dtype=np.float32)

    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01)
    model.fit(X)

    rows = []

    for alpha in [0.8, 0.5]:
        r10, n10, r50, n50 = [], [], [], []

        for u in users_eval:
            ids, scores = model.recommend(
                userid=u,
                user_items=X[u],
                N=200,
                filter_already_liked_items=True
            )

            profile = user_profiles[u]
            rescored = []

            for item_id, als_score in zip(ids, scores):
                sim = cosine(profile, item_emb[int(item_id)])
                final_score = alpha * float(als_score) + (1 - alpha) * sim
                rescored.append((int(item_id), final_score))

            rescored.sort(key=lambda x: x[1], reverse=True)
            rec = [item for item, _ in rescored]

            gts = gt[u]
            r10.append(recall_at_k(rec, gts, 10))
            n10.append(ndcg_at_k(rec, gts, 10))
            r50.append(recall_at_k(rec, gts, 50))
            n50.append(ndcg_at_k(rec, gts, 50))

        rows.append({
            "model": "als_rerank",
            "content": args.content_name,
            "integration": f"rerank_alpha_{alpha}",
            "recall@10": float(np.nanmean(r10)),
            "ndcg@10": float(np.nanmean(n10)),
            "recall@50": float(np.nanmean(r50)),
            "ndcg@50": float(np.nanmean(n50)),
            "users_eval": len(users_eval)
        })

    if os.path.exists(OUT_FILE):
        df = pd.read_csv(OUT_FILE)
        df = df[df["content"] != args.content_name]
        df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
    else:
        df = pd.DataFrame(rows)

    df.to_csv(OUT_FILE, index=False)
    print(pd.DataFrame(rows))

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/train_eval_rerank_generic.py \
  --emb_path /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_metadata_features.npy \
  --content_name metadata

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:12<00:00,  1.19it/s]
        model  ... users_eval
0  als_rerank  ...       2416
1  als_rerank  ...       2416

[2 rows x 8 columns]


In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/train_eval_rerank_generic.py \
  --emb_path /content/drive/MyDrive/vk_lsvd_project/data/processed/local_item_fused_embeddings.npy \
  --content_name fused

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:06<00:00,  2.16it/s]
        model  ... users_eval
0  als_rerank  ...       2416
1  als_rerank  ...       2416

[2 rows x 8 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv

model,content,integration,recall@10,ndcg@10,recall@50,ndcg@50,users_eval
popularity,none,none,0.0256622516556291,0.0138803553558226,0.0658112582781457,0.0224615017608284,2416
als,none,none,0.0049668874172185,0.0028696195857278,0.0211092715231788,0.0063504354043472,2416
bpr,none,none,0.0004139072847682,0.0001474367496308,0.0016556291390728,0.000448857830686,2416
als_rerank,item_emb,rerank_alpha_0.8,0.0062086092715231,0.0030564599806902,0.023592715231788,0.0068119108936735,2416
als_rerank,item_emb,rerank_alpha_0.5,0.0062086092715231,0.0030691444261068,0.0231788079470198,0.0067542211991691,2416
als_rerank,metadata,rerank_alpha_0.8,0.0053807947019867,0.0029829537949199,0.0248344370860927,0.0072023273894442,2416
als_rerank,metadata,rerank_alpha_0.5,0.0053807947019867,0.0029829537949199,0.0252483443708609,0.0072828963466985,2416
als_rerank,fused,rerank_alpha_0.8,0.00869205298013245,0.004592074906736957,0.028973509933774833,0.008948863003738698,2416
als_rerank,fused,rerank_alpha_0.5,0.0086920

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/make_plots.py

Plots saved to: /content/drive/MyDrive/vk_lsvd_project/outputs/plots


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/outputs/plots/

total 192K
-rw------- 1 root root 67K Feb 25 18:42 ndcg10_bar.png
-rw------- 1 root root 65K Feb 25 18:42 recall50_bar.png
-rw------- 1 root root 60K Feb 25 18:42 robustness_bar.png


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/eval_robustness.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_robustness.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def cosine(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def build_gt(df):
    gt = {}
    if len(df) == 0:
        return gt
    for u, g in df.groupby("u"):
        gt[int(u)] = list(map(int, g["i"].values))
    return gt

def eval_subset(users_subset, gt, rec_fn):
    r10, n10, r50, n50 = [], [], [], []
    for u in users_subset:
        if u not in gt:
            continue
        gts = set(gt[u])
        rec = rec_fn(u)
        r10.append(recall_at_k(rec, gts, 10))
        n10.append(ndcg_at_k(rec, gts, 10))
        r50.append(recall_at_k(rec, gts, 50))
        n50.append(ndcg_at_k(rec, gts, 50))
    if len(r10) == 0:
        return None
    return {
        "recall@10": float(np.nanmean(r10)),
        "ndcg@10": float(np.nanmean(n10)),
        "recall@50": float(np.nanmean(r50)),
        "ndcg@50": float(np.nanmean(n50)),
        "users_eval": len(r10)
    }

def main():
    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
    fused = np.load(os.path.join(DATA_DIR, "local_item_fused_embeddings.npy"))

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    # GT
    gt = build_gt(test)

    # Histories
    user_hist = train.groupby("u").size().to_dict()
    item_pop = train.groupby("i").size().to_dict()

    # Popularity
    pop = np.asarray(X.sum(axis=0)).ravel()
    pop_rank = np.argsort(-pop)

    def recommend_pop(u, k=100):
        seen = set(train.loc[train["u"] == u, "i"].tolist())
        rec = [i for i in pop_rank if i not in seen]
        return rec[:k]

    # ALS
    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01)
    model.fit(X)

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = list(map(int, g["i"].values))

    user_profiles = {}
    for u, items in hist.items():
        vecs = fused[items]
        user_profiles[u] = vecs.mean(axis=0) if len(vecs) else np.zeros(fused.shape[1], dtype=np.float32)

    def recommend_fused_rerank(u, k=100, alpha=0.8):
        ids, scores = model.recommend(
            userid=u,
            user_items=X[u],
            N=200,
            filter_already_liked_items=True
        )
        profile = user_profiles[u]
        rescored = []
        for item_id, als_score in zip(ids, scores):
            sim = cosine(profile, fused[int(item_id)])
            final_score = alpha * float(als_score) + (1 - alpha) * sim
            rescored.append((int(item_id), final_score))
        rescored.sort(key=lambda x: x[1], reverse=True)
        return [item for item, _ in rescored][:k]

    rows = []

    # ---------- short-history ----------
    short_users = [u for u, cnt in user_hist.items() if cnt <= 2 and u in gt]
    for model_name, fn in [
        ("popularity", lambda u: recommend_pop(u)),
        ("als_rerank_fused", lambda u: recommend_fused_rerank(u)),
    ]:
        res = eval_subset(short_users, gt, fn)
        if res:
            rows.append({
                "scenario": "short_history_users",
                "model": model_name,
                **res
            })

    # ---------- rare-items ----------
    rare_users = []
    for u, items in gt.items():
        if any(item_pop.get(i, 0) <= 2 for i in items):
            rare_users.append(u)

    for model_name, fn in [
        ("popularity", lambda u: recommend_pop(u)),
        ("als_rerank_fused", lambda u: recommend_fused_rerank(u)),
    ]:
        res = eval_subset(rare_users, gt, fn)
        if res:
            rows.append({
                "scenario": "rare_items_users",
                "model": model_name,
                **res
            })

    df = pd.DataFrame(rows)
    df.to_csv(OUT_FILE, index=False)
    print(df)

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/eval_robustness.py

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:13<00:00,  1.08it/s]
              scenario  ... users_eval
0  short_history_users  ...       2416
1  short_history_users  ...       2416
2     rare_items_users  ...       1394
3     rare_items_users  ...       1394

[4 rows x 7 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_robustness.csv

scenario,model,recall@10,ndcg@10,recall@50,ndcg@50,users_eval
short_history_users,popularity,0.02566225165562914,0.013880355355822693,0.0658112582781457,0.02246150176082847,2416
short_history_users,als_rerank_fused,0.008278145695364239,0.004238886055908992,0.027731788079470198,0.008373785284629771,2416
rare_items_users,popularity,0.0,0.0,0.0,0.0,1394
rare_items_users,als_rerank_fused,0.0,0.0,0.0,0.0,1394


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/make_robustness_plot.py << 'PY'
import os
os.environ["MPLBACKEND"] = "Agg"

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

inp = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_robustness.csv"
out = "/content/drive/MyDrive/vk_lsvd_project/outputs/plots/robustness_bar.png"

df = pd.read_csv(inp)
df["label"] = df["scenario"] + "|" + df["model"]

plt.figure()
plt.bar(df["label"], df["ndcg@10"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("ndcg@10")
plt.tight_layout()
plt.savefig(out, dpi=150)
print("saved:", out)
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/make_robustness_plot.py

saved: /content/drive/MyDrive/vk_lsvd_project/outputs/plots/robustness_bar.png


In [ ]:
!ls -lh /content/drive/MyDrive/vk_lsvd_project/outputs/plots/robustness_bar.png

-rw------- 1 root root 60K Feb 25 18:42 /content/drive/MyDrive/vk_lsvd_project/outputs/plots/robustness_bar.png


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/train_eval_lightfm.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from lightfm import LightFM

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def build_gt(df):
    gt = {}
    if len(df) == 0:
        return gt
    for u, g in df.groupby("u"):
        gt[int(u)] = set(map(int, g["i"].values))
    return gt

def main():
    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
    item_features_dense = np.load(os.path.join(DATA_DIR, "local_item_metadata_features.npy")).astype(np.float32)

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    # LightFM expects sparse item features
    item_features = csr_matrix(item_features_dense)

    gt = build_gt(test)
    users_eval = sorted(gt.keys())
    if len(users_eval) == 0:
        print("No users in test after mapping. Stop.")
        return

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = set(map(int, g["i"].values))

    model = LightFM(no_components=64, loss="warp", learning_rate=0.05, random_state=42)
    model.fit(X, item_features=item_features, epochs=15, num_threads=2)

    r10, n10, r50, n50 = [], [], [], []

    all_items = np.arange(n_items)

    for u in users_eval:
        scores = model.predict(u, all_items, item_features=item_features, num_threads=2)
        seen = hist.get(u, set())

        order = np.argsort(-scores)
        rec = [int(i) for i in order if int(i) not in seen][:100]

        gts = gt[u]
        r10.append(recall_at_k(rec, gts, 10))
        n10.append(ndcg_at_k(rec, gts, 10))
        r50.append(recall_at_k(rec, gts, 50))
        n50.append(ndcg_at_k(rec, gts, 50))

    row = {
        "model": "lightfm",
        "content": "metadata",
        "integration": "hybrid",
        "recall@10": float(np.nanmean(r10)),
        "ndcg@10": float(np.nanmean(n10)),
        "recall@50": float(np.nanmean(r50)),
        "ndcg@50": float(np.nanmean(n50)),
        "users_eval": len(users_eval)
    }

    if os.path.exists(OUT_FILE):
        df = pd.read_csv(OUT_FILE)
        df = df[df["model"] != "lightfm"]
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    df.to_csv(OUT_FILE, index=False)
    print(pd.DataFrame([row]))

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/train_eval_lightfm.py

     model  ... users_eval
0  lightfm  ...       2416

[1 rows x 8 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_main.csv

model,content,integration,recall@10,ndcg@10,recall@50,ndcg@50,users_eval
popularity,none,none,0.0256622516556291,0.0138803553558226,0.0658112582781457,0.0224615017608284,2416
als,none,none,0.0049668874172185,0.0028696195857278,0.0211092715231788,0.0063504354043472,2416
bpr,none,none,0.0004139072847682,0.0001474367496308,0.0016556291390728,0.000448857830686,2416
als_rerank,item_emb,rerank_alpha_0.8,0.0062086092715231,0.0030564599806902,0.023592715231788,0.0068119108936735,2416
als_rerank,item_emb,rerank_alpha_0.5,0.0062086092715231,0.0030691444261068,0.0231788079470198,0.0067542211991691,2416
als_rerank,metadata,rerank_alpha_0.8,0.0053807947019867,0.0029829537949199,0.0248344370860927,0.0072023273894442,2416
als_rerank,metadata,rerank_alpha_0.5,0.0053807947019867,0.0029829537949199,0.0252483443708609,0.0072828963466985,2416
als_rerank,fused,rerank_alpha_0.8,0.0086920529801324,0.0045920749067369,0.0289735099337748,0.0089488630037386,2416
als_rerank,fused,rerank_alpha_0.5,0.00869205298013

In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/eval_best_model_extra_metrics.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_best_extra.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def mrr_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            return 1.0 / rank
    return 0.0

def cosine(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def build_gt(df):
    gt = {}
    for u, g in df.groupby("u"):
        gt[int(u)] = set(map(int, g["i"].values))
    return gt

def main():
    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
    fused = np.load(os.path.join(DATA_DIR, "local_item_fused_embeddings.npy"))

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    gt = build_gt(test)
    users_eval = sorted(gt.keys())

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = list(map(int, g["i"].values))

    user_profiles = {}
    for u, items in hist.items():
        vecs = fused[items]
        user_profiles[u] = vecs.mean(axis=0) if len(vecs) else np.zeros(fused.shape[1], dtype=np.float32)

    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01)
    model.fit(X)

    alpha = 0.8
    r10, n10, r50, n50, mrr10 = [], [], [], [], []

    for u in users_eval:
        ids, scores = model.recommend(
            userid=u,
            user_items=X[u],
            N=200,
            filter_already_liked_items=True
        )

        profile = user_profiles[u]
        rescored = []
        for item_id, als_score in zip(ids, scores):
            sim = cosine(profile, fused[int(item_id)])
            final_score = alpha * float(als_score) + (1 - alpha) * sim
            rescored.append((int(item_id), final_score))

        rescored.sort(key=lambda x: x[1], reverse=True)
        rec = [item for item, _ in rescored]

        gts = gt[u]
        r10.append(recall_at_k(rec, gts, 10))
        n10.append(ndcg_at_k(rec, gts, 10))
        r50.append(recall_at_k(rec, gts, 50))
        n50.append(ndcg_at_k(rec, gts, 50))
        mrr10.append(mrr_at_k(rec, gts, 10))

    df = pd.DataFrame([{
        "model": "als_rerank_fused_best",
        "alpha": alpha,
        "recall@10": float(np.nanmean(r10)),
        "ndcg@10": float(np.nanmean(n10)),
        "recall@50": float(np.nanmean(r50)),
        "ndcg@50": float(np.nanmean(n50)),
        "mrr@10": float(np.nanmean(mrr10)),
        "users_eval": len(users_eval)
    }])

    df.to_csv(OUT_FILE, index=False)
    print(df)

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/eval_best_model_extra_metrics.py

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:11<00:00,  1.26it/s]
                   model  ...  users_eval
0  als_rerank_fused_best  ...        2416

[1 rows x 8 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_best_extra.csv

model,alpha,recall@10,ndcg@10,recall@50,ndcg@50,mrr@10,users_eval
als_rerank_fused_best,0.8,0.0074503311258278145,0.0038291596264039434,0.027731788079470198,0.008172755945884513,0.002738686534216336,2416


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/sweep_alpha_fused.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_alpha_sweep.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def cosine(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def build_gt(df):
    gt = {}
    for u, g in df.groupby("u"):
        gt[int(u)] = set(map(int, g["i"].values))
    return gt

def main():
    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
    fused = np.load(os.path.join(DATA_DIR, "local_item_fused_embeddings.npy"))

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    gt = build_gt(test)
    users_eval = sorted(gt.keys())

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = list(map(int, g["i"].values))

    user_profiles = {}
    for u, items in hist.items():
        vecs = fused[items]
        user_profiles[u] = vecs.mean(axis=0) if len(vecs) else np.zeros(fused.shape[1], dtype=np.float32)

    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01)
    model.fit(X)

    rows = []
    for alpha in [0.2, 0.35, 0.5, 0.65, 0.8]:
        r10, n10, r50, n50 = [], [], [], []

        for u in users_eval:
            ids, scores = model.recommend(
                userid=u,
                user_items=X[u],
                N=200,
                filter_already_liked_items=True
            )

            profile = user_profiles[u]
            rescored = []
            for item_id, als_score in zip(ids, scores):
                sim = cosine(profile, fused[int(item_id)])
                final_score = alpha * float(als_score) + (1 - alpha) * sim
                rescored.append((int(item_id), final_score))

            rescored.sort(key=lambda x: x[1], reverse=True)
            rec = [item for item, _ in rescored]

            gts = gt[u]
            r10.append(recall_at_k(rec, gts, 10))
            n10.append(ndcg_at_k(rec, gts, 10))
            r50.append(recall_at_k(rec, gts, 50))
            n50.append(ndcg_at_k(rec, gts, 50))

        rows.append({
            "alpha": alpha,
            "recall@10": float(np.nanmean(r10)),
            "ndcg@10": float(np.nanmean(n10)),
            "recall@50": float(np.nanmean(r50)),
            "ndcg@50": float(np.nanmean(n50)),
            "users_eval": len(users_eval)
        })

    df = pd.DataFrame(rows)
    df.to_csv(OUT_FILE, index=False)
    print(df.sort_values("ndcg@10", ascending=False))

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/sweep_alpha_fused.py

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:09<00:00,  1.59it/s]
   alpha  ...  users_eval
0   0.20  ...        2416
1   0.35  ...        2416
2   0.50  ...        2416
3   0.65  ...        2416
4   0.80  ...        2416

[5 rows x 6 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_alpha_sweep.csv

alpha,recall@10,ndcg@10,recall@50,ndcg@50,users_eval
0.2,0.00869205298013245,0.0045860000299154475,0.026490066225165563,0.00845259479507462,2416
0.35,0.00869205298013245,0.00456786106019738,0.026490066225165563,0.008434455825356551,2416
0.5,0.00869205298013245,0.00456786106019738,0.026490066225165563,0.008435656630466982,2416
0.65,0.00869205298013245,0.004539167582578752,0.026490066225165563,0.008408785815597316,2416
0.8,0.00869205298013245,0.004529699927870685,0.026076158940397352,0.008325931279882532,2416


In [ ]:
%%bash
cat > /content/drive/MyDrive/vk_lsvd_project/src/eval_robustness_v2.py << 'PY'
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

DATA_DIR = "/content/drive/MyDrive/vk_lsvd_project/data/processed"
OUT_FILE = "/content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_robustness_v2.csv"

def recall_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    return sum((i in gt_set) for i in rec[:k]) / len(gt_set)

def ndcg_at_k(rec, gt_set, k):
    if not gt_set:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(rec[:k], start=1):
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)
    m = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, m + 1))
    return dcg / idcg if idcg > 0 else np.nan

def cosine(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def build_gt(df):
    gt = {}
    for u, g in df.groupby("u"):
        gt[int(u)] = list(map(int, g["i"].values))
    return gt

def eval_subset(users_subset, gt, rec_fn):
    r10, n10 = [], []
    for u in users_subset:
        if u not in gt:
            continue
        gts = set(gt[u])
        rec = rec_fn(u)
        r10.append(recall_at_k(rec, gts, 10))
        n10.append(ndcg_at_k(rec, gts, 10))
    if len(r10) == 0:
        return None
    return {
        "recall@10": float(np.nanmean(r10)),
        "ndcg@10": float(np.nanmean(n10)),
        "users_eval": len(r10)
    }

def main():
    train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
    test = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
    fused = np.load(os.path.join(DATA_DIR, "local_item_fused_embeddings.npy"))

    n_users = int(train["u"].max()) + 1
    n_items = int(train["i"].max()) + 1

    X = csr_matrix(
        (np.ones(len(train), dtype=np.float32), (train["u"].values, train["i"].values)),
        shape=(n_users, n_items)
    )

    gt = build_gt(test)
    user_hist = train.groupby("u").size().to_dict()
    item_pop = train.groupby("i").size().to_dict()

    pop = np.asarray(X.sum(axis=0)).ravel()
    pop_rank = np.argsort(-pop)

    def recommend_pop(u, k=100):
        seen = set(train.loc[train["u"] == u, "i"].tolist())
        rec = [i for i in pop_rank if i not in seen]
        return rec[:k]

    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01)
    model.fit(X)

    hist = {}
    for u, g in train.groupby("u"):
        hist[int(u)] = list(map(int, g["i"].values))

    user_profiles = {}
    for u, items in hist.items():
        vecs = fused[items]
        user_profiles[u] = vecs.mean(axis=0) if len(vecs) else np.zeros(fused.shape[1], dtype=np.float32)

    def recommend_fused(u, k=100, alpha=0.8):
        ids, scores = model.recommend(
            userid=u,
            user_items=X[u],
            N=200,
            filter_already_liked_items=True
        )
        profile = user_profiles[u]
        rescored = []
        for item_id, als_score in zip(ids, scores):
            na = np.linalg.norm(profile)
            nb = np.linalg.norm(fused[int(item_id)])
            sim = 0.0 if (na == 0 or nb == 0) else float(np.dot(profile, fused[int(item_id)]) / (na * nb))
            final_score = alpha * float(als_score) + (1 - alpha) * sim
            rescored.append((int(item_id), final_score))
        rescored.sort(key=lambda x: x[1], reverse=True)
        return [item for item, _ in rescored][:k]

    rows = []

    # short-history <= 2
    short_users = [u for u, cnt in user_hist.items() if cnt <= 2 and u in gt]
    for model_name, fn in [
        ("popularity", lambda u: recommend_pop(u)),
        ("als_rerank_fused", lambda u: recommend_fused(u)),
    ]:
        res = eval_subset(short_users, gt, fn)
        if res:
            rows.append({"scenario": "short_history_le_2", "model": model_name, **res})

    # rare-items <= 5
    rare_users = []
    for u, items in gt.items():
        if any(item_pop.get(i, 0) <= 5 for i in items):
            rare_users.append(u)

    for model_name, fn in [
        ("popularity", lambda u: recommend_pop(u)),
        ("als_rerank_fused", lambda u: recommend_fused(u)),
    ]:
        res = eval_subset(rare_users, gt, fn)
        if res:
            rows.append({"scenario": "rare_items_le_5", "model": model_name, **res})

    df = pd.DataFrame(rows)
    df.to_csv(OUT_FILE, index=False)
    print(df)

if __name__ == "__main__":
    main()
PY

In [ ]:
!./bin/micromamba run -n recsys python /content/drive/MyDrive/vk_lsvd_project/src/eval_robustness_v2.py

/root/.local/share/mamba/envs/recsys/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100% 15/15 [00:09<00:00,  1.56it/s]
             scenario  ... users_eval
0  short_history_le_2  ...       2416
1  short_history_le_2  ...       2416
2     rare_items_le_5  ...       1850
3     rare_items_le_5  ...       1850

[4 rows x 5 columns]


In [ ]:
!cat /content/drive/MyDrive/vk_lsvd_project/outputs/metrics/results_robustness_v2.csv

scenario,model,recall@10,ndcg@10,users_eval
short_history_le_2,popularity,0.02566225165562914,0.013880355355822693,2416
short_history_le_2,als_rerank_fused,0.007864238410596027,0.004243116191809542,2416
rare_items_le_5,popularity,0.0,0.0,1850
rare_items_le_5,als_rerank_fused,0.0,0.0,1850
